# setup

In [ ]:
# =========================
# 1. Setup
# =========================
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_PATH = "/content/drive/MyDrive/Deep Learning Project/models/ntu60_hrnet.pkl"


# =========================
# 2. Load Data
# =========================
with open(MODEL_PATH, 'rb') as f:
    data = pickle.load(f)

LR=0.1
EPOCHS=20
NUM_WORKERS=2
NUM_CLASSES=60
BATCH_SIZE=16

# CTR-GCN
- LR=0.1
- EPOCHS=20
- NUM_WORKERS=2
- NUM_CLASSES=60  
- BATCH_SIZE=16
- Train Loss: 0.9134 | Train Acc: 70.07%
- Val Loss:   0.7641 | Val Acc:   74.68%
- Final Test Loss (Best Model): 0.7641
- Final Test Accuracy (Best Model): 74.68%

In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import shutil

class NTUDataset(Dataset):
    def __init__(self, data_path, split_name='xsub_train', max_frames=64, num_joints=17, max_persons=2, is_training=True):
        self.data_path = data_path
        self.split_name = split_name
        self.max_frames = max_frames
        self.num_joints = num_joints
        self.max_persons = max_persons
        self.is_training = is_training

        print(f"Loading data from {data_path} for split {split_name}...")
        with open(data_path, 'rb') as f:
            self.data = pickle.load(f)

        self.split_ids = set(self.data['split'][split_name])

        self.samples = []
        for ann in tqdm(self.data['annotations'], desc=f"Filtering {split_name}"):
            if ann['frame_dir'] in self.split_ids:
                self.samples.append(ann)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ann = self.samples[idx]
        kp = ann['keypoint'].copy() # (M, T, V, C)
        score = ann.get('keypoint_score', None)
        if score is not None:
            score = score.copy()

        M, T, V, C = kp.shape

        # Temporal Augmentation / Padding
        if T >= self.max_frames:
            if self.is_training:
                start = np.random.randint(0, T - self.max_frames + 1)
            else:
                start = (T - self.max_frames) // 2
            kp = kp[:, start:start+self.max_frames, :, :]
            if score is not None:
                score = score[:, start:start+self.max_frames, :]
        else:
            pad_len = self.max_frames - T
            pad_kp = np.zeros((M, pad_len, V, C), dtype=kp.dtype)
            kp = np.concatenate([kp, pad_kp], axis=1)
            if score is not None:
                pad_score = np.zeros((M, pad_len, V), dtype=score.dtype)
                score = np.concatenate([score, pad_score], axis=1)

        # Persons Padding / Truncation
        if M < self.max_persons:
            pad_m = self.max_persons - M
            pad_kp = np.zeros((pad_m, self.max_frames, V, C), dtype=kp.dtype)
            kp = np.concatenate([kp, pad_kp], axis=0)
            if score is not None:
                pad_score = np.zeros((pad_m, self.max_frames, V), dtype=score.dtype)
                score = np.concatenate([score, pad_score], axis=0)
        elif M > self.max_persons:
            kp = kp[:self.max_persons]
            if score is not None:
                score = score[:self.max_persons]

        # Normalization (Center to first valid person's first valid frame nose)
        valid_mask = (kp.sum(axis=-1) != 0)
        if valid_mask.any():
            m_idx, t_idx, _ = np.where(valid_mask)
            first_m, first_t = m_idx[0], t_idx[0]
            center_joint = kp[first_m, first_t, 0, :].copy()
            mask = np.expand_dims(valid_mask, axis=-1)
            kp = kp - center_joint * mask

        # --- Stronger Augmentations (Scaling and Temporal Masking) ---
        if self.is_training:
            # 1. Spatial Scaling
            scale = np.random.uniform(0.8, 1.2)
            kp[:, :, :, :2] = kp[:, :, :, :2] * scale

            # 2. Temporal Masking (Mask 10% of the frames randomly)
            num_mask = int(self.max_frames * 0.1)
            mask_indices = np.random.choice(self.max_frames, num_mask, replace=False)
            kp[:, mask_indices, :, :] = 0
            if score is not None:
                score[:, mask_indices, :] = 0

        # Add score as third channel if available
        if score is not None:
            score = np.expand_dims(score, axis=-1)
            kp = np.concatenate([kp, score], axis=-1)

        kp = kp.transpose((3, 1, 2, 0)) # Transpose to (C, T, V, M)
        return torch.tensor(kp, dtype=torch.float32), torch.tensor(ann['label'], dtype=torch.long)

# Graph Topology Definition
def get_hop_distance(num_node, edge, max_hop=1):
    A = np.zeros((num_node, num_node))
    for i, j in edge:
        A[j, i] = 1
        A[i, j] = 1

    hop_dis = np.zeros((num_node, num_node)) + np.inf
    transfer_mat = [np.linalg.matrix_power(A, d) for d in range(max_hop + 1)]
    arrive_mat = (np.stack(transfer_mat) > 0)
    for d in range(max_hop, -1, -1):
        hop_dis[arrive_mat[d]] = d
    return hop_dis

class Graph():
    def __init__(self, num_node=17, center=0):
        self.num_node = num_node
        self.center = center
        self.inward = [(1, 0), (2, 0), (3, 1), (4, 2), (5, 0), (6, 0), (7, 5), (8, 6), (9, 7), (10, 8), (11, 5), (12, 6), (13, 11), (14, 12), (15, 13), (16, 14)]
        self.outward = [(j, i) for (i, j) in self.inward]
        self.edge = self.inward + self.outward
        self.hop_dis = get_hop_distance(self.num_node, self.edge, max_hop=1)
        self.A = self.get_spatial_graph()

    def get_spatial_graph(self):
        A = np.zeros((3, self.num_node, self.num_node))
        for i in range(self.num_node):
            for j in range(self.num_node):
                if self.hop_dis[j, i] == 0:
                    A[0, j, i] = 1
                elif self.hop_dis[j, i] == 1:
                    if self.hop_dis[j, self.center] < self.hop_dis[i, self.center]:
                        A[1, j, i] = 1
                    else:
                        A[2, j, i] = 1
        for i in range(3):
            D = A[i].sum(axis=1)
            D[D == 0] = 1
            D = D ** -1
            A[i] = A[i] * D.reshape(-1, 1)
        return A

# CTR-GCN Blocks
class CTRGC(nn.Module):
    def __init__(self, in_channels, out_channels, rel_reduction=8):
        super(CTRGC, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.rel_channels = in_channels // rel_reduction if in_channels >= 16 else 8

        self.conv1 = nn.Conv2d(self.in_channels, self.rel_channels, kernel_size=1)
        self.conv2 = nn.Conv2d(self.in_channels, self.rel_channels, kernel_size=1)
        self.conv3 = nn.Conv2d(self.in_channels, self.out_channels, kernel_size=1)
        self.conv4 = nn.Conv2d(self.rel_channels, self.out_channels, kernel_size=1)
        self.tanh = nn.Tanh()
        self.alpha = nn.Parameter(torch.zeros(1))

    def forward(self, x, A):
        # x: (N, C, T, V)
        x1 = self.conv1(x).mean(-2) # (N, rel_C, V)
        x2 = self.conv2(x).mean(-2)
        x3 = self.conv3(x)          # (N, out_C, T, V)

        x1 = x1.unsqueeze(-1)       # (N, rel_C, V, 1)
        x2 = x2.unsqueeze(-2)       # (N, rel_C, 1, V)

        # Topology Attention
        attn = self.tanh(x1 - x2)   # (N, rel_C, V, V)
        attn = self.conv4(attn)     # (N, out_C, V, V)

        adj = A.unsqueeze(0).unsqueeze(0) + self.alpha * attn # (N, out_C, V, V)
        out = torch.einsum('n c v w, n c t w -> n c t v', adj, x3)
        return out

class CTRGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, residual=True, num_subset=3):
        super(CTRGCNBlock, self).__init__()
        self.num_subset = num_subset

        # Make sure subset out_channels sum up exactly to out_channels
        subset_out_channels = [out_channels // num_subset] * num_subset
        for i in range(out_channels % num_subset):
            subset_out_channels[i] += 1

        self.ctr_gc = nn.ModuleList([
            CTRGC(in_channels, subset_out_channels[i]) for i in range(num_subset)
        ])

        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_channels),
        )

        if not residual:
            self.residual = lambda x: 0
        elif (in_channels == out_channels) and (stride == 1):
            self.residual = lambda x: x
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels)
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, A):
        # A: (3, V, V)
        res = self.residual(x)

        y = []
        for i in range(self.num_subset):
            y.append(self.ctr_gc[i](x, A[i]))

        x = torch.cat(y, dim=1) # (N, out_channels, T, V)
        x = self.tcn(x)
        return self.relu(x + res)

class CTRGCN(nn.Module):
    def __init__(self, in_channels, num_classes, num_node=17):
        super(CTRGCN, self).__init__()
        self.graph = Graph(num_node)
        A = torch.tensor(self.graph.A, dtype=torch.float32, requires_grad=False)
        self.register_buffer('A', A)

        self.data_bn = nn.BatchNorm1d(in_channels * num_node)

        self.networks = nn.ModuleList((
            CTRGCNBlock(in_channels, 64, residual=False),
            CTRGCNBlock(64, 64),
            CTRGCNBlock(64, 64),
            CTRGCNBlock(64, 64),
            CTRGCNBlock(64, 128, stride=2),
            CTRGCNBlock(128, 128),
            CTRGCNBlock(128, 128),
            CTRGCNBlock(128, 256, stride=2),
            CTRGCNBlock(256, 256),
            CTRGCNBlock(256, 256),
        ))

        self.fcn = nn.Conv2d(256, num_classes, kernel_size=1)

    def forward(self, x):
        # x: (N, C, T, V, M)
        N, C, T, V, M = x.size()
        x = x.permute(0, 4, 3, 1, 2).contiguous() # (N, M, V, C, T)
        x = x.view(N * M, V * C, T)
        x = self.data_bn(x)
        x = x.view(N, M, V, C, T)
        x = x.permute(0, 1, 3, 4, 2).contiguous() # (N, M, C, T, V)
        x = x.view(N * M, C, T, V)

        for block in self.networks:
            x = block(x, self.A)

        x = F.avg_pool2d(x, x.size()[2:]) # (N*M, 256, 1, 1)
        x = x.view(N, M, -1, 1, 1).mean(dim=1) # (N, 256, 1, 1)

        x = self.fcn(x)
        x = x.view(N, -1)
        return x

# Plotting and Metrics
def plot_curves(history, output_dir):
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'training_curves.png'))
    plt.close()

def plot_confusion(y_true, y_pred, output_dir):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(24, 20)) # Large size to accommodate 60 classes
    sns.heatmap(cm, annot=False, cmap='Blues')
    plt.title('Confusion Matrix')
    plt.ylabel('True Class')
    plt.xlabel('Predicted Class')
    plt.savefig(os.path.join(output_dir, 'confusion_matrix.png'))
    plt.close()

# Training and Evaluation Wrappers
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, targets in tqdm(dataloader, desc="Training", leave=False):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    return total_loss / len(dataloader), 100. * correct / total

def validate(model, dataloader, criterion, device, phase="Validation"):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in tqdm(dataloader, desc=phase, leave=False):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    return total_loss / len(dataloader), 100. * correct / total

def evaluate_test(model, dataloader, criterion, device, output_dir):
    model.eval()
    all_preds = []
    all_targets = []
    total_loss, correct, total = 0, 0, 0

    print("\n--- Starting Test Evaluation ---")
    with torch.no_grad():
        for inputs, targets in tqdm(dataloader, desc="Testing (xview_val)"):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            total_loss += loss.item()
            _, preds = outputs.max(1)
            total += targets.size(0)
            correct += preds.eq(targets).sum().item()

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.numpy())

    test_loss = total_loss / len(dataloader)
    test_acc = 100. * correct / total

    # Compute metrics
    print("\nClassification Report:")
    print(classification_report(all_targets, all_preds))
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

    plot_confusion(all_targets, all_preds, output_dir)
    print(f"Confusion matrix saved to '{output_dir}/confusion_matrix.png'")
    return test_loss, test_acc

def main():
    # Setup Paths
    MODEL_PATH = "/content/drive/MyDrive/Deep Learning Project/models/ntu60_hrnet.pkl"
    output_dir = "./output"
    os.makedirs(output_dir, exist_ok=True)
    saved_model_path = os.path.join(os.path.dirname(MODEL_PATH), 'ctr_gcn_checkpoint.pth')
    best_model_path = os.path.join(os.path.dirname(MODEL_PATH), 'ctr_gcn_best.pth')

    # Check if we are running locally instead of colab, adjust path if necessary
    if not os.path.exists(MODEL_PATH) and os.path.exists('models/ntu60_hrnet.pkl'):
        print(f"Path {MODEL_PATH} not found. Falling back to local 'models/ntu60_hrnet.pkl'")
        MODEL_PATH = 'models/ntu60_hrnet.pkl'

    # Hyperparameters
    num_classes = NUM_CLASSES
    batch_size = BATCH_SIZE
    epochs = EPOCHS
    lr = LR
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Executing on device: {device}")

    # Datasets
    # We use xview_train for training and xview_val for testing/val since user requested xview_val for test
    train_dataset = NTUDataset(MODEL_PATH, split_name='xview_train', is_training=True)
    val_dataset = NTUDataset(MODEL_PATH, split_name='xview_val', is_training=False)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS) # Same as val here

    # Instantiate Model
    sample_kp, _ = train_dataset[0]
    in_channels = sample_kp.size(0)
    model = CTRGCN(in_channels=in_channels, num_classes=num_classes).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=0.0001)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 40], gamma=0.1)

    # History Tracking & Checkpoint Resume
    start_epoch = 0
    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    if os.path.exists(saved_model_path):
        print(f"Loading existing checkpoint from '{saved_model_path}'...")
        checkpoint = torch.load(saved_model_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_acc = checkpoint.get('best_val_acc', 0.0)
        history = checkpoint.get('history', history)
        print(f"Resumed from epoch {start_epoch} with Best Validation Acc: {best_val_acc:.2f}%")

    # Training Loop
    for epoch in range(start_epoch, epochs):
        print(f"\n--- Epoch {epoch+1}/{epochs} ---")
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        scheduler.step()

        # Track history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")

        # Checkpointing
        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc

        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_acc': best_val_acc,
            'history': history
        }

        torch.save(checkpoint, saved_model_path)
        if is_best:
            print("=> Saving new best model")
            shutil.copyfile(saved_model_path, best_model_path)

        # Plot curves continuously
        plot_curves(history, output_dir)

    # --- Final Test Evaluation ---
    print("\nLoading Best Model for Final Evaluation...")
    if os.path.exists(best_model_path):
        model.load_state_dict(torch.load(best_model_path, map_location=device)['model_state_dict'])

    evaluate_test(model, test_loader, criterion, device, output_dir)
    print("Run completed successfully!")

In [ ]:
main()

Executing on device: cuda
Loading data from /content/drive/MyDrive/Deep Learning Project/models/ntu60_hrnet.pkl for split xview_train...


Filtering xview_train: 100%|██████████| 56578/56578 [00:00<00:00, 1390963.52it/s]


Loading data from /content/drive/MyDrive/Deep Learning Project/models/ntu60_hrnet.pkl for split xview_val...


Filtering xview_val: 100%|██████████| 56578/56578 [00:00<00:00, 882733.82it/s]



--- Epoch 1/20 ---


...

Train Loss: 0.9017 | Train Acc: 70.49%
Val Loss:   0.7624 | Val Acc:   73.97%

--- Epoch 19/20 ---


Train Loss: 0.8867 | Train Acc: 70.90%
Val Loss:   1.0307 | Val Acc:   66.35%

--- Epoch 20/20 ---


Train Loss: 0.8766 | Train Acc: 71.48%
Val Loss:   0.8315 | Val Acc:   72.72%

Loading Best Model for Final Evaluation...

--- Starting Test Evaluation ---


Testing (xview_val): 100%|██████████| 1184/1184 [00:50<00:00, 23.32it/s]



Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.76      0.77       316
           1       0.57      0.84      0.68       316
           2       0.83      0.48      0.61       316
           3       0.87      0.52      0.65       316
           4       0.73      0.74      0.73       316
           5       0.75      0.96      0.84       316
           6       0.90      0.93      0.91       316
           7       0.74      0.99      0.85       315
           8       1.00      0.99      1.00       316
           9       0.64      0.43      0.51       316
          10       0.48      0.23      0.31       315
          11       0.57      0.03      0.05       315
          12       0.62      0.75      0.68       316
          13       0.82      0.86      0.84       316
          14       0.88      0.81      0.84       316
          15       0.55      0.73      0.63       315
          16       0.81      0.19      0.30       316
   